## Trạng thái bản lưu trên GitHub

Tiền xử lý đã dùng trong quá trình nghiên cứu; cần video private để chạy.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Đang chạy bằng CPU")

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import re
import zipfile

INPUT_ROOT = Path("/kaggle/input")
EXTRACT_ROOT = Path("/kaggle/working/gmdcsa24")

print("Các dataset đã gắn vào notebook:")
for path in INPUT_ROOT.iterdir():
    print("-", path)

# Tìm video trực tiếp
videos = sorted(INPUT_ROOT.rglob("*.mp4"))

print("\nSố video tìm thấy trực tiếp:", len(videos))

# Nếu không thấy video, kiểm tra và giải nén ZIP
if not videos:
    zip_files = sorted(INPUT_ROOT.rglob("*.zip"))

    print("Số file ZIP tìm thấy:", len(zip_files))

    if zip_files:
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

        for zip_path in zip_files:
            print("Đang giải nén:", zip_path)

            with zipfile.ZipFile(zip_path, "r") as archive:
                archive.extractall(EXTRACT_ROOT)

        videos = sorted(EXTRACT_ROOT.rglob("*.mp4"))
        print("Số video sau giải nén:", len(videos))

In [ ]:
from pathlib import Path
import pandas as pd
import re

INPUT_ROOT = Path("/kaggle/input")
videos = sorted(INPUT_ROOT.rglob("*.mp4"))

records = []

for video_path in videos:
    # Video nằm trực tiếp trong thư mục ADL hoặc Fall
    label_folder = video_path.parent.name.strip().upper()

    if label_folder not in {"ADL", "FALL"}:
        print("Không nhận diện được nhãn:", video_path)
        continue

    # Tìm thư mục Subject 1, Subject 2, Subject 3 hoặc Subject 4
    subject_id = None

    for parent_path in video_path.parents:
        match = re.search(
            r"subject[\s_-]*(\d+)",
            parent_path.name,
            flags=re.IGNORECASE,
        )

        if match:
            subject_id = int(match.group(1))
            break

    if subject_id is None:
        print("Không nhận diện được subject:", video_path)
        continue

    label = label_folder

    records.append({
        "video_id": f"S{subject_id}_{label}_{video_path.stem}",
        "subject": subject_id,
        "label": label,
        "label_id": 1 if label == "FALL" else 0,
        "filename": video_path.name,
        "path": str(video_path),
    })

manifest = pd.DataFrame(records)

print("Tổng video tìm thấy:", len(videos))
print("Tổng video hợp lệ:", len(manifest))
print("\nPhân bố theo subject và label:")
print(manifest.groupby(["subject", "label"]).size())

display(manifest.head())

assert len(manifest) == 160, (
    f"Mong đợi 160 video nhưng chỉ nhận diện được {len(manifest)}"
)

manifest.to_csv(
    "/kaggle/working/gmdcsa24_manifest.csv",
    index=False,
)

print("\nĐã lưu:")
print("/kaggle/working/gmdcsa24_manifest.csv")

In [ ]:
import cv2
import pandas as pd
from tqdm.auto import tqdm

# Đọc danh sách 160 video đã tạo ở cell trước
manifest_path = "/kaggle/working/gmdcsa24_manifest.csv"
manifest = pd.read_csv(manifest_path)

# Nơi lưu thông tin kiểm tra của từng video
video_info = []

# Duyệt lần lượt toàn bộ video trong manifest
for row in tqdm(
    manifest.itertuples(index=False),
    total=len(manifest),
    desc="Kiểm tra video",
):
    # Mở video bằng OpenCV
    capture = cv2.VideoCapture(row.path)
    readable = capture.isOpened()

    if readable:
        # Lấy FPS, số frame và độ phân giải của video
        fps = float(capture.get(cv2.CAP_PROP_FPS))
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    else:
        # Đặt giá trị bằng 0 nếu video không mở được
        fps = 0.0
        frame_count = 0
        width = 0
        height = 0

    # Thời lượng video = tổng số frame / số frame mỗi giây
    duration = frame_count / fps if fps > 0 else 0.0

    video_info.append({
        "video_id": row.video_id,
        "readable": readable,
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_seconds": duration,
    })

    # Giải phóng video sau khi đọc xong
    capture.release()

# Chuyển kết quả kiểm tra thành DataFrame
video_info = pd.DataFrame(video_info)

# Ghép thông tin kỹ thuật vào manifest ban đầu
manifest = manifest.merge(
    video_info,
    on="video_id",
    how="left",
)

# In kết quả tổng quát
print("Tổng video:", len(manifest))
print("Video đọc được:", int(manifest["readable"].sum()))
print("Video lỗi:", int((~manifest["readable"]).sum()))

# Thống kê FPS của toàn bộ video
print("\nThống kê FPS:")
display(manifest["fps"].describe())

# Thống kê thời lượng video
print("\nThống kê thời lượng:")
display(manifest["duration_seconds"].describe())

# Đếm số video theo từng độ phân giải
print("\nCác độ phân giải:")
resolution_summary = (
    manifest.groupby(["width", "height"])
    .size()
    .reset_index(name="video_count")
    .sort_values("video_count", ascending=False)
)

display(resolution_summary)

# Tìm các video bị lỗi
broken_videos = manifest.loc[
    ~manifest["readable"],
    ["video_id", "path"],
]

if broken_videos.empty:
    print("\nTất cả video đều đọc được.")
else:
    print("\nCác video không đọc được:")
    display(broken_videos)

# Ghi thêm FPS, số frame và thời lượng vào manifest
manifest.to_csv(manifest_path, index=False)

print("\nĐã cập nhật:", manifest_path)

# 4. Trích xuất pose thử nghiệm

In [ ]:
import mediapipe as mp

# Kiểm tra MediaPipe có import được không
print("MediaPipe version:", mp.__version__)

# Kiểm tra riêng module Pose Landmarker
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("Pose Landmarker: OK")

In [ ]:
from pathlib import Path
import urllib.request

# Nơi lưu model trong phiên làm việc Kaggle
MODEL_PATH = Path(
    "/kaggle/working/pose_landmarker_full.task"
)

# Model Pose Landmarker Full chính thức
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_full/float16/latest/"
    "pose_landmarker_full.task"
)

# Chỉ tải khi file chưa tồn tại
if not MODEL_PATH.exists():
    print("Đang tải Pose Landmarker Full...")

    urllib.request.urlretrieve(
        MODEL_URL,
        MODEL_PATH,
    )

# Kiểm tra file sau khi tải
if not MODEL_PATH.exists():
    raise RuntimeError("Không tải được model.")

model_size_mb = MODEL_PATH.stat().st_size / (1024 * 1024)

print("Model tồn tại:", MODEL_PATH.exists())
print("Đường dẫn:", MODEL_PATH)
print("Dung lượng:", round(model_size_mb, 2), "MB")

In [ ]:
import mediapipe as mp

# Cấu hình model để phát hiện tối đa một người
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=str(MODEL_PATH)
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_poses=1,
)

# Thử khởi tạo Pose Landmarker
with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as detector:
    print("Khởi tạo Pose Landmarker thành công.")

In [ ]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt

# Đọc lại danh sách video
manifest = pd.read_csv(
    "/kaggle/working/gmdcsa24_manifest.csv"
)

# Chọn video Fall đầu tiên
sample = manifest[
    manifest["label"].str.upper() == "FALL"
].iloc[0]

video_path = sample["path"]

# Mở video
capture = cv2.VideoCapture(video_path)

frame_count = int(
    capture.get(cv2.CAP_PROP_FRAME_COUNT)
)

# Chọn frame ở giữa video
frame_index = frame_count // 2
capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)

success, frame_bgr = capture.read()
capture.release()

if not success:
    raise RuntimeError("Không đọc được frame thử nghiệm.")

# OpenCV dùng BGR, MediaPipe và matplotlib dùng RGB
frame_rgb = cv2.cvtColor(
    frame_bgr,
    cv2.COLOR_BGR2RGB,
)

print("Video:", sample["video_id"])
print("Nhãn:", sample["label"])
print("Frame:", frame_index, "/", frame_count)
print("Kích thước:", frame_rgb.shape)

plt.figure(figsize=(12, 7))
plt.imshow(frame_rgb)
plt.title(f"{sample['video_id']} — frame {frame_index}")
plt.axis("off")
plt.show()

In [ ]:
import mediapipe as mp

# Cấu hình phát hiện tối đa một người
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=str(MODEL_PATH)
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
)

# Chuyển ảnh NumPy sang định dạng MediaPipe
mediapipe_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=frame_rgb,
)

# Nhận diện pose
with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as pose_detector:
    pose_result = pose_detector.detect(mediapipe_image)

pose_count = len(pose_result.pose_landmarks)

print("Số người phát hiện:", pose_count)

if pose_count > 0:
    landmarks = pose_result.pose_landmarks[0]

    print("Số điểm khớp:", len(landmarks))
else:
    print("Không phát hiện được người trong frame này.")

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Các cạnh chính nối 33 điểm khớp
POSE_CONNECTIONS = [
    # Đầu nối với vai
    (0, 11),
    (0, 12),

    # Hai vai và hai tay
    (11, 12),
    (11, 13),
    (13, 15),
    (12, 14),
    (14, 16),

    # Thân người
    (11, 23),
    (12, 24),
    (23, 24),

    # Chân trái
    (23, 25),
    (25, 27),
    (27, 29),
    (29, 31),
    (27, 31),

    # Chân phải
    (24, 26),
    (26, 28),
    (28, 30),
    (30, 32),
    (28, 32),
]

# Tạo bản sao để không sửa ảnh gốc
annotated_frame = frame_rgb.copy()

if pose_count > 0:
    landmarks = pose_result.pose_landmarks[0]
    height, width = annotated_frame.shape[:2]

    pixel_points = []

    # Chuyển tọa độ chuẩn hóa sang tọa độ pixel
    for landmark in landmarks:
        x = int(landmark.x * width)
        y = int(landmark.y * height)

        pixel_points.append((x, y))

    # Vẽ các đường nối giữa khớp
    for start_index, end_index in POSE_CONNECTIONS:
        start = pixel_points[start_index]
        end = pixel_points[end_index]

        cv2.line(
            annotated_frame,
            start,
            end,
            color=(0, 255, 0),
            thickness=3,
        )

    # Vẽ các điểm khớp
    for point in pixel_points:
        cv2.circle(
            annotated_frame,
            point,
            radius=5,
            color=(255, 0, 0),
            thickness=-1,
        )

plt.figure(figsize=(12, 7))
plt.imshow(annotated_frame)
plt.title(f"MediaPipe Pose — {sample['video_id']}")
plt.axis("off")
plt.show()

In [ ]:
def extract_pose_sequence(video_path):
    """
    Trích xuất pose liên tục trên toàn video,
    sau đó lấy mẫu xuống 32 frame.
    """

    capture = cv2.VideoCapture(video_path)

    if not capture.isOpened():
        raise RuntimeError(
            f"Không mở được video: {video_path}"
        )

    frame_count = int(
        capture.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        capture.get(cv2.CAP_PROP_FPS)
    )

    if frame_count <= 0 or fps <= 0:
        capture.release()
        raise RuntimeError(
            f"Video không hợp lệ: {video_path}"
        )

    # Dùng NaN để phân biệt frame mất pose với tọa độ thật
    full_sequence = np.full(
        (
            frame_count,
            NUM_LANDMARKS,
            NUM_FEATURES,
        ),
        np.nan,
        dtype=np.float32,
    )

    detected_mask = np.zeros(
        frame_count,
        dtype=bool,
    )

    # Hạ nhẹ ngưỡng vì video có tư thế cúi và nằm
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(MODEL_PATH)
        ),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.35,
        min_pose_presence_confidence=0.35,
        min_tracking_confidence=0.35,
    )

    frame_index = 0
    previous_timestamp = -1

    with mp.tasks.vision.PoseLandmarker.create_from_options(
        options
    ) as detector:

        while True:
            success, frame_bgr = capture.read()

            if not success:
                break

            frame_rgb = cv2.cvtColor(
                frame_bgr,
                cv2.COLOR_BGR2RGB,
            )

            mediapipe_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=frame_rgb,
            )

            # Timestamp phải tăng dần trong VIDEO mode
            timestamp_ms = int(
                round(frame_index * 1000 / fps)
            )

            timestamp_ms = max(
                timestamp_ms,
                previous_timestamp + 1,
            )

            previous_timestamp = timestamp_ms

            result = detector.detect_for_video(
                mediapipe_image,
                timestamp_ms,
            )

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]

                for joint_index, landmark in enumerate(
                    landmarks
                ):
                    full_sequence[
                        frame_index,
                        joint_index,
                    ] = [
                        landmark.x,
                        landmark.y,
                        landmark.z,
                        landmark.visibility,
                    ]

                detected_mask[frame_index] = True

            frame_index += 1

    capture.release()

    # Một số video có số frame thực tế khác metadata
    full_sequence = full_sequence[:frame_index]
    detected_mask = detected_mask[:frame_index]

    raw_detection_rate = detected_mask.mean()

    # Không có pose ở bất kỳ frame nào
    if not detected_mask.any():
        empty_sequence = np.zeros(
            (
                NUM_FRAMES,
                NUM_LANDMARKS,
                NUM_FEATURES,
            ),
            dtype=np.float32,
        )

        empty_mask = np.zeros(
            NUM_FRAMES,
            dtype=bool,
        )

        return empty_sequence, empty_mask, 0.0

    # Nội suy các frame bị mất pose
    valid_indices = np.where(detected_mask)[0]
    all_indices = np.arange(frame_index)

    for joint_index in range(NUM_LANDMARKS):
        for feature_index in range(NUM_FEATURES):
            valid_values = full_sequence[
                valid_indices,
                joint_index,
                feature_index,
            ]

            full_sequence[
                :,
                joint_index,
                feature_index,
            ] = np.interp(
                all_indices,
                valid_indices,
                valid_values,
            )

    # Chuẩn hóa mọi video về đúng 32 frame
    selected_indices = np.linspace(
        0,
        frame_index - 1,
        NUM_FRAMES,
    ).round().astype(int)

    sampled_sequence = full_sequence[
        selected_indices
    ]

    sampled_mask = detected_mask[
        selected_indices
    ]

    return (
        sampled_sequence.astype(np.float32),
        sampled_mask,
        float(raw_detection_rate),
    )

In [ ]:
test_sequence, test_mask, raw_rate = (
    extract_pose_sequence(sample["path"])
)

print("Video:", sample["video_id"])
print("Kích thước pose:", test_sequence.shape)

print(
    "Tỷ lệ phát hiện trên toàn bộ frame:",
    f"{raw_rate:.2%}",
)

print(
    "Frame mẫu phát hiện trực tiếp:",
    int(test_mask.sum()),
    "/",
    NUM_FRAMES,
)

assert test_sequence.shape == (32, 33, 4)
assert not np.isnan(test_sequence).any()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Đọc manifest nếu biến manifest chưa còn trong bộ nhớ
manifest = pd.read_csv(
    "/kaggle/working/gmdcsa24_manifest.csv"
)

# Chuẩn hóa tên nhãn để tránh khác biệt Fall/FALL
manifest["label"] = manifest["label"].str.upper()

# Chọn 4 trường hợp đại diện
test_cases = [
    (1, "ADL"),
    (2, "FALL"),
    (3, "ADL"),
    (4, "FALL"),
]

quality_results = []

for subject_id, label in tqdm(
    test_cases,
    desc="Kiểm tra chất lượng pose",
):
    # Chọn video đầu tiên đúng subject và nhãn
    candidates = manifest[
        (manifest["subject"] == subject_id)
        & (manifest["label"] == label)
    ]

    if candidates.empty:
        print(
            f"Không tìm thấy Subject {subject_id} - {label}"
        )
        continue

    sample_row = candidates.iloc[0]

    # Trích xuất pose bằng hàm đã sửa
    sequence, sampled_mask, raw_rate = (
        extract_pose_sequence(sample_row["path"])
    )

    # Kiểm tra chuỗi sau nội suy có hợp lệ không
    contains_nan = bool(
        np.isnan(sequence).any()
    )

    quality_results.append({
        "video_id": sample_row["video_id"],
        "subject": subject_id,
        "label": label,
        "raw_detection_rate": raw_rate,
        "sampled_detected_frames": int(
            sampled_mask.sum()
        ),
        "sampled_total_frames": NUM_FRAMES,
        "contains_nan": contains_nan,
    })

quality_df = pd.DataFrame(quality_results)

# Hiển thị tỷ lệ dưới dạng phần trăm
quality_df["detection_percent"] = (
    quality_df["raw_detection_rate"] * 100
).round(2)

display(
    quality_df[
        [
            "video_id",
            "subject",
            "label",
            "detection_percent",
            "sampled_detected_frames",
            "sampled_total_frames",
            "contains_nan",
        ]
    ]
)

In [ ]:
# Vẽ tỷ lệ nhận diện pose của 4 video
plt.figure(figsize=(10, 5))

bars = plt.bar(
    quality_df["video_id"],
    quality_df["detection_percent"],
    color=[
        "#3498db" if label == "ADL" else "#e74c3c"
        for label in quality_df["label"]
    ],
)

# Mốc tham khảo 80%
plt.axhline(
    80,
    color="orange",
    linestyle="--",
    label="Mốc 80%",
)

# Ghi giá trị trên từng cột
for bar, value in zip(
    bars,
    quality_df["detection_percent"],
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.1f}%",
        ha="center",
    )

plt.ylim(0, 105)
plt.ylabel("Tỷ lệ phát hiện pose (%)")
plt.xlabel("Video")
plt.title("Kiểm tra chất lượng MediaPipe Pose")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

# Lưu kết quả QA
quality_path = (
    "/kaggle/working/gmdcsa24_pose_quality_check.csv"
)

quality_df.to_csv(
    quality_path,
    index=False,
)

print("Đã lưu:", quality_path)

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Đọc và sắp xếp manifest để thứ tự dữ liệu cố định
manifest = pd.read_csv(
    "/kaggle/working/gmdcsa24_manifest.csv"
).sort_values(
    ["subject", "label_id", "filename"]
).reset_index(drop=True)

# Thư mục lưu riêng kết quả của từng video
CACHE_DIR = Path(
    "/kaggle/working/gmdcsa24_pose_cache"
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_progress.csv"
)

ERROR_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_errors.csv"
)

progress_records = []
error_records = []

for row in tqdm(
    manifest.itertuples(index=False),
    total=len(manifest),
    desc="Trích xuất pose 160 video",
):
    # Mỗi video có một file cache riêng
    cache_path = CACHE_DIR / f"{row.video_id}.npz"

    try:
        if cache_path.exists():
            # Không xử lý lại video đã hoàn thành
            cached = np.load(cache_path)

            sequence = cached["sequence"]
            sampled_mask = cached["sampled_mask"]
            raw_rate = float(cached["raw_detection_rate"])
            source = "cache"

        else:
            # Trích xuất pose bằng hàm đã kiểm tra
            sequence, sampled_mask, raw_rate = (
                extract_pose_sequence(row.path)
            )

            # Kiểm tra output trước khi lưu
            if sequence.shape != (
                NUM_FRAMES,
                NUM_LANDMARKS,
                NUM_FEATURES,
            ):
                raise ValueError(
                    f"Sai kích thước pose: {sequence.shape}"
                )

            if sampled_mask.shape != (NUM_FRAMES,):
                raise ValueError(
                    f"Sai kích thước mask: {sampled_mask.shape}"
                )

            if not np.isfinite(sequence).all():
                raise ValueError(
                    "Pose còn chứa NaN hoặc Inf"
                )

            # Lưu ngay sau mỗi video để tránh mất tiến độ
            np.savez_compressed(
                cache_path,
                sequence=sequence.astype(np.float32),
                sampled_mask=sampled_mask.astype(bool),
                raw_detection_rate=np.float32(raw_rate),
            )

            source = "processed"

        progress_records.append({
            "video_id": row.video_id,
            "subject": row.subject,
            "label": row.label,
            "raw_detection_rate": raw_rate,
            "sampled_detected_frames": int(
                sampled_mask.sum()
            ),
            "status": "success",
            "source": source,
            "cache_path": str(cache_path),
        })

    except Exception as error:
        # Ghi lại lỗi nhưng vẫn tiếp tục các video sau
        error_records.append({
            "video_id": row.video_id,
            "path": row.path,
            "error": str(error),
        })

        progress_records.append({
            "video_id": row.video_id,
            "subject": row.subject,
            "label": row.label,
            "raw_detection_rate": 0.0,
            "sampled_detected_frames": 0,
            "status": "failed",
            "source": "error",
            "cache_path": str(cache_path),
        })

        print(
            f"\nLỗi {row.video_id}: {error}"
        )

    # Cập nhật tiến độ sau mỗi video
    pd.DataFrame(progress_records).to_csv(
        PROGRESS_PATH,
        index=False,
    )

    # Giải phóng bộ nhớ tạm
    gc.collect()

# Lưu danh sách lỗi nếu có
if error_records:
    pd.DataFrame(error_records).to_csv(
        ERROR_PATH,
        index=False,
    )

print("\nHoàn thành batch.")
print("Thành công:", len(progress_records) - len(error_records))
print("Lỗi:", len(error_records))
print("Cache:", CACHE_DIR)
print("Tiến độ:", PROGRESS_PATH)

if error_records:
    print("Danh sách lỗi:", ERROR_PATH)
    

In [ ]:
progress_df = pd.read_csv(
    "/kaggle/working/gmdcsa24_pose_progress.csv"
)

successful_df = progress_df[
    progress_df["status"] == "success"
].copy()

print("Tổng số video:", len(progress_df))
print("Thành công:", len(successful_df))
print(
    "Thất bại:",
    int((progress_df["status"] == "failed").sum()),
)

print("\nThống kê tỷ lệ phát hiện:")
display(
    (
        successful_df["raw_detection_rate"] * 100
    ).describe()
)

# Liệt kê các video có tỷ lệ dưới 80%
low_quality = successful_df[
    successful_df["raw_detection_rate"] < 0.80
].copy()

if low_quality.empty:
    print("\nKhông có video nào dưới 80%.")
else:
    low_quality["detection_percent"] = (
        low_quality["raw_detection_rate"] * 100
    ).round(2)

    print("\nCác video có tỷ lệ dưới 80%:")

    display(
        low_quality[
            [
                "video_id",
                "subject",
                "label",
                "detection_percent",
                "sampled_detected_frames",
            ]
        ].sort_values("detection_percent")
    )

In [ ]:
if len(successful_df) != len(manifest):
    raise RuntimeError(
        "Chưa đủ 160 video thành công. "
        "Cần xử lý các video lỗi trước khi gộp."
    )

all_sequences = []
all_masks = []
all_detection_rates = []

# Đọc cache theo đúng thứ tự trong manifest
for row in tqdm(
    manifest.itertuples(index=False),
    total=len(manifest),
    desc="Gộp dữ liệu pose",
):
    cache_path = CACHE_DIR / f"{row.video_id}.npz"
    cached = np.load(cache_path)

    all_sequences.append(
        cached["sequence"].astype(np.float32)
    )

    all_masks.append(
        cached["sampled_mask"].astype(bool)
    )

    all_detection_rates.append(
        float(cached["raw_detection_rate"])
    )

# Ghép thành các mảng lớn
poses = np.stack(all_sequences)
masks = np.stack(all_masks)
detection_rates = np.asarray(
    all_detection_rates,
    dtype=np.float32,
)

labels = manifest[
    "label_id"
].to_numpy(dtype=np.int64)

subjects = manifest[
    "subject"
].to_numpy(dtype=np.int64)

video_ids = manifest[
    "video_id"
].astype(str).to_numpy()

OUTPUT_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_32.npz"
)

# Lưu dữ liệu cuối cùng
np.savez_compressed(
    OUTPUT_PATH,
    poses=poses,
    masks=masks,
    labels=labels,
    subjects=subjects,
    video_ids=video_ids,
    detection_rates=detection_rates,
)

# Thêm tỷ lệ phát hiện vào manifest
manifest["pose_detection_rate"] = detection_rates

POSE_MANIFEST_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_manifest.csv"
)

manifest.to_csv(
    POSE_MANIFEST_PATH,
    index=False,
)

print("Kích thước poses:", poses.shape)
print("Kích thước masks:", masks.shape)
print("Kích thước labels:", labels.shape)
print("Kích thước subjects:", subjects.shape)

print(
    "Tỷ lệ phát hiện trung bình:",
    f"{detection_rates.mean():.2%}",
)

print(
    "Tỷ lệ phát hiện thấp nhất:",
    f"{detection_rates.min():.2%}",
)

print("\nĐã lưu:", OUTPUT_PATH)
print("Đã lưu:", POSE_MANIFEST_PATH)

In [ ]:
from pathlib import Path
import shutil
import numpy as np

NPZ_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_32.npz"
)

BACKUP_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_32_backup.npz"
)

TEMP_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_32_fixed.npz"
)

# Tạo bản sao trước khi sửa
if not BACKUP_PATH.exists():
    shutil.copy2(NPZ_PATH, BACKUP_PATH)
    print("Đã tạo bản dự phòng:", BACKUP_PATH)

# allow_pickle=True chỉ dùng vì đây là file do notebook tự tạo
with np.load(NPZ_PATH, allow_pickle=True) as old_data:
    arrays = {
        key: old_data[key]
        for key in old_data.files
    }

print("Các mảng trong file:", list(arrays.keys()))
print("Kiểu video_ids cũ:", arrays["video_ids"].dtype)

# Chuyển object array sang Unicode array an toàn
arrays["video_ids"] = np.asarray(
    arrays["video_ids"],
    dtype="U",
)

print("Kiểu video_ids mới:", arrays["video_ids"].dtype)

# Lưu thành file tạm
np.savez_compressed(
    TEMP_PATH,
    **arrays,
)

# Thay file cũ bằng file đã sửa
TEMP_PATH.replace(NPZ_PATH)

print("Đã sửa:", NPZ_PATH)

In [ ]:
import numpy as np

NPZ_PATH = "/kaggle/working/gmdcsa24_pose_32.npz"

# Không cần allow_pickle=True nữa
with np.load(NPZ_PATH) as pose_data:
    print("Các mảng:", pose_data.files)

    poses = pose_data["poses"]
    masks = pose_data["masks"]
    labels = pose_data["labels"]
    subjects = pose_data["subjects"]
    video_ids = pose_data["video_ids"]
    detection_rates = pose_data["detection_rates"]

print("\nKích thước poses:", poses.shape)
print("Kích thước masks:", masks.shape)
print("Kích thước labels:", labels.shape)
print("Kích thước subjects:", subjects.shape)
print("Kích thước video_ids:", video_ids.shape)

print("\nKiểu video_ids:", video_ids.dtype)
print("Video đầu tiên:", video_ids[0])

print("\nCó NaN:", np.isnan(poses).any())
print("Có Inf:", np.isinf(poses).any())

assert poses.shape == (160, 32, 33, 4)
assert masks.shape == (160, 32)
assert labels.shape == (160,)
assert subjects.shape == (160,)
assert video_ids.shape == (160,)
assert video_ids.dtype.kind in {"U", "S"}
assert np.isfinite(poses).all()

print("\nDữ liệu pose hợp lệ và không cần pickle.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

POSE_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_32.npz"
)

MANIFEST_PATH = Path(
    "/kaggle/working/gmdcsa24_pose_manifest.csv"
)

# Đọc dữ liệu pose
with np.load(POSE_PATH) as data:
    poses = data["poses"].astype(np.float32)
    masks = data["masks"].astype(bool)
    labels = data["labels"].astype(np.int64)
    subjects = data["subjects"].astype(np.int64)
    video_ids = data["video_ids"].astype("U")
    detection_rates = data["detection_rates"].astype(
        np.float32
    )

pose_manifest = pd.read_csv(MANIFEST_PATH)

# Căn manifest theo đúng thứ tự video trong NPZ
pose_manifest = (
    pose_manifest
    .set_index("video_id")
    .loc[video_ids]
    .reset_index()
)

# Thời lượng được dùng để tính vận tốc theo giây
durations = pose_manifest[
    "duration_seconds"
].to_numpy(dtype=np.float32)

print("Poses:", poses.shape)
print("Thời lượng:", durations.shape)
print("Thời lượng nhỏ nhất:", durations.min())

In [ ]:
NUM_SAMPLES, NUM_FRAMES, NUM_JOINTS, _ = poses.shape

# Tọa độ và độ tin cậy
xyz = poses[..., :3].copy()
visibility = np.clip(
    poses[..., 3:4],
    0.0,
    1.0,
)

# MediaPipe landmark indices
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_HIP = 23
RIGHT_HIP = 24

relative_xyz = np.zeros_like(xyz)
root_position = np.zeros(
    (NUM_SAMPLES, NUM_FRAMES, 3),
    dtype=np.float32,
)

body_scales = np.zeros(
    NUM_SAMPLES,
    dtype=np.float32,
)

torso_angles = np.zeros(
    (NUM_SAMPLES, NUM_FRAMES),
    dtype=np.float32,
)

bbox_widths = np.zeros_like(torso_angles)
bbox_heights = np.zeros_like(torso_angles)
bbox_ratios = np.zeros_like(torso_angles)

for sample_index in range(NUM_SAMPLES):
    sample_xyz = xyz[sample_index]

    # Tâm hông đại diện cho gốc cơ thể
    hip_center = (
        sample_xyz[:, LEFT_HIP]
        + sample_xyz[:, RIGHT_HIP]
    ) / 2.0

    shoulder_center = (
        sample_xyz[:, LEFT_SHOULDER]
        + sample_xyz[:, RIGHT_SHOULDER]
    ) / 2.0

    root_position[sample_index] = hip_center

    # Chiều dài thân dùng làm tỷ lệ cơ thể
    torso_lengths = np.linalg.norm(
        shoulder_center[:, :2] - hip_center[:, :2],
        axis=1,
    )

    valid_lengths = torso_lengths[
        torso_lengths > 1e-3
    ]

    body_scale = (
        np.median(valid_lengths)
        if len(valid_lengths) > 0
        else 1.0
    )

    body_scale = max(float(body_scale), 1e-3)
    body_scales[sample_index] = body_scale

    # Skeleton tương đối: lấy tâm hông làm gốc
    relative_xyz[sample_index] = (
        sample_xyz - hip_center[:, None, :]
    ) / body_scale

    # Góc nghiêng thân, chuẩn hóa về khoảng gần [-1, 1]
    torso_vector = shoulder_center - hip_center

    torso_angles[sample_index] = (
        np.arctan2(
            torso_vector[:, 0],
            -torso_vector[:, 1],
        )
        / np.pi
    )

    # Bounding box từ các điểm khớp
    minimum_xy = sample_xyz[..., :2].min(axis=1)
    maximum_xy = sample_xyz[..., :2].max(axis=1)

    bbox_width = maximum_xy[:, 0] - minimum_xy[:, 0]
    bbox_height = maximum_xy[:, 1] - minimum_xy[:, 1]

    bbox_widths[sample_index] = bbox_width
    bbox_heights[sample_index] = bbox_height

    bbox_ratios[sample_index] = (
        bbox_width / np.maximum(bbox_height, 1e-6)
    )

In [ ]:
# Khoảng thời gian giữa hai frame mẫu của từng video
delta_times = durations / (NUM_FRAMES - 1)
delta_times = np.maximum(delta_times, 1e-6)

joint_velocity = np.zeros_like(relative_xyz)
joint_acceleration = np.zeros_like(relative_xyz)

root_velocity = np.zeros_like(root_position)
root_acceleration = np.zeros_like(root_position)

for sample_index in range(NUM_SAMPLES):
    delta_time = delta_times[sample_index]

    # Vận tốc khớp theo thời gian thực của video
    joint_velocity[sample_index, 1:] = (
        np.diff(
            relative_xyz[sample_index],
            axis=0,
        )
        / delta_time
    )

    # Gia tốc khớp
    joint_acceleration[sample_index, 1:] = (
        np.diff(
            joint_velocity[sample_index],
            axis=0,
        )
        / delta_time
    )

    # Vận tốc tâm hông
    root_velocity[sample_index, 1:] = (
        np.diff(
            root_position[sample_index],
            axis=0,
        )
        / delta_time
    )

    # Gia tốc tâm hông
    root_acceleration[sample_index, 1:] = (
        np.diff(
            root_velocity[sample_index],
            axis=0,
        )
        / delta_time
    )

# Nhánh skeleton: tọa độ tương đối và visibility
skeleton_features = np.concatenate(
    [
        relative_xyz,
        visibility,
    ],
    axis=-1,
)

# Nhánh motion theo từng khớp
motion_features = np.concatenate(
    [
        joint_velocity,
        joint_acceleration,
    ],
    axis=-1,
)

# Dùng thuận tiện cho các baseline đơn giản
combined_joint_features = np.concatenate(
    [
        skeleton_features,
        motion_features,
    ],
    axis=-1,
)

# Đặc trưng chuyển động toàn thân
global_features = np.concatenate(
    [
        root_position,
        root_velocity,
        root_acceleration,
        torso_angles[..., None],
        bbox_widths[..., None],
        bbox_heights[..., None],
        bbox_ratios[..., None],
    ],
    axis=-1,
)

print("Skeleton:", skeleton_features.shape)
print("Motion:", motion_features.shape)
print("Combined:", combined_joint_features.shape)
print("Global:", global_features.shape)

In [ ]:
arrays_to_check = {
    "skeleton_features": skeleton_features,
    "motion_features": motion_features,
    "combined_joint_features": combined_joint_features,
    "global_features": global_features,
    "body_scales": body_scales,
}

for name, array in arrays_to_check.items():
    print(
        name,
        "| shape:",
        array.shape,
        "| finite:",
        np.isfinite(array).all(),
    )

    if not np.isfinite(array).all():
        raise ValueError(
            f"{name} chứa NaN hoặc Inf"
        )

FEATURE_PATH = Path(
    "/kaggle/working/gmdcsa24_features_32.npz"
)

global_feature_names = np.asarray(
    [
        "root_x",
        "root_y",
        "root_z",
        "root_velocity_x",
        "root_velocity_y",
        "root_velocity_z",
        "root_acceleration_x",
        "root_acceleration_y",
        "root_acceleration_z",
        "torso_angle",
        "bbox_width",
        "bbox_height",
        "bbox_ratio",
    ],
    dtype="U",
)

np.savez_compressed(
    FEATURE_PATH,
    skeleton_features=skeleton_features.astype(
        np.float32
    ),
    motion_features=motion_features.astype(
        np.float32
    ),
    combined_joint_features=combined_joint_features.astype(
        np.float32
    ),
    global_features=global_features.astype(
        np.float32
    ),
    masks=masks,
    labels=labels,
    subjects=subjects,
    video_ids=video_ids,
    detection_rates=detection_rates,
    durations=durations,
    body_scales=body_scales,
    global_feature_names=global_feature_names,
)

print("\nĐã lưu:", FEATURE_PATH)
print(
    "Dung lượng:",
    round(FEATURE_PATH.stat().st_size / 1024**2, 2),
    "MB",
)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Đọc manifest pose theo đúng thứ tự video
pose_manifest = pd.read_csv(
    "/kaggle/working/gmdcsa24_pose_manifest.csv"
)

pose_manifest = pose_manifest.reset_index(
    drop=True
)

# Đảm bảo manifest có đúng 160 video
assert len(pose_manifest) == 160

subjects = sorted(
    pose_manifest["subject"].unique()
)

print("Các subject:", subjects)
print("Số subject:", len(subjects))

fold_records = []

for test_subject in subjects:
    # Chọn subject kế tiếp làm validation
    # Ví dụ: test 1 -> validation 2
    test_position = subjects.index(test_subject)
    val_position = (
        test_position + 1
    ) % len(subjects)

    val_subject = subjects[val_position]

    train_subjects = [
        subject
        for subject in subjects
        if subject not in {
            test_subject,
            val_subject,
        }
    ]

    for sample_index, row in pose_manifest.iterrows():
        current_subject = int(row["subject"])

        if current_subject == test_subject:
            split = "test"
        elif current_subject == val_subject:
            split = "val"
        else:
            split = "train"

        fold_records.append({
            "fold": f"test_subject_{test_subject}",
            "sample_index": sample_index,
            "video_id": row["video_id"],
            "subject": current_subject,
            "label": row["label"],
            "label_id": int(row["label_id"]),
            "split": split,
            "test_subject": test_subject,
            "val_subject": val_subject,
            "train_subjects": ",".join(
                map(str, train_subjects)
            ),
        })

folds_df = pd.DataFrame(fold_records)

print("Kích thước folds:", folds_df.shape)
display(folds_df.head())

In [ ]:
for fold_name, fold_data in folds_df.groupby(
    "fold"
):
    train_subjects = set(
        fold_data.loc[
            fold_data["split"] == "train",
            "subject",
        ]
    )

    val_subjects = set(
        fold_data.loc[
            fold_data["split"] == "val",
            "subject",
        ]
    )

    test_subjects = set(
        fold_data.loc[
            fold_data["split"] == "test",
            "subject",
        ]
    )

    # Ba tập phải tách biệt hoàn toàn
    assert train_subjects.isdisjoint(
        val_subjects
    )
    assert train_subjects.isdisjoint(
        test_subjects
    )
    assert val_subjects.isdisjoint(
        test_subjects
    )

    print(f"\n{fold_name}")
    print("Train subjects:", sorted(train_subjects))
    print("Val subject:", sorted(val_subjects))
    print("Test subject:", sorted(test_subjects))

    print("\nSố mẫu:")
    print(
        fold_data.groupby(
            ["split", "label"]
        ).size()
    )

In [ ]:
FOLDS_PATH = Path(
    "/kaggle/working/gmdcsa24_cross_subject_folds.csv"
)

folds_df.to_csv(
    FOLDS_PATH,
    index=False,
)

print("Đã lưu:", FOLDS_PATH)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

required_files = [
    Path("/kaggle/working/gmdcsa24_pose_32.npz"),
    Path("/kaggle/working/gmdcsa24_features_32.npz"),
    Path("/kaggle/working/gmdcsa24_pose_manifest.csv"),
    Path("/kaggle/working/gmdcsa24_cross_subject_folds.csv"),
]

# Kiểm tra các file cần thiết
for file_path in required_files:
    print(
        file_path.name,
        "->",
        "OK" if file_path.exists() else "MISSING",
    )

assert all(
    file_path.exists()
    for file_path in required_files
), "Thiếu file đầu ra cần thiết."

# Kiểm tra thứ tự video giữa features và folds
with np.load(
    "/kaggle/working/gmdcsa24_features_32.npz"
) as data:
    video_ids = data["video_ids"].astype("U")
    labels = data["labels"].astype(np.int64)
    subjects = data["subjects"].astype(np.int64)

folds_df = pd.read_csv(
    "/kaggle/working/gmdcsa24_cross_subject_folds.csv"
)

sample_indices = folds_df[
    "sample_index"
].to_numpy(dtype=np.int64)

fold_video_ids = folds_df[
    "video_id"
].astype(str).to_numpy()

# Mỗi sample_index phải trỏ đúng video trong NPZ
alignment_ok = np.array_equal(
    video_ids[sample_indices],
    fold_video_ids,
)

print("\nSố mẫu:", len(video_ids))
print("Số fold:", folds_df["fold"].nunique())
print("Căn chỉnh sample_index:", alignment_ok)

assert len(video_ids) == 160
assert folds_df["fold"].nunique() == 4
assert alignment_ok

print("\nDữ liệu sẵn sàng cho Motion-SVM.")